# Évaluation finale : la version `ml_14` sur le test, lu une seule fois

**But** : vérifier, sur des clients que rien n'a jamais vu, ce que la version finale du machine learning tient vraiment, et présenter le résultat comme la banque l'utiliserait. Le test (`data/ML/test_dataset.csv`, 20 % des clients laissés au ML, découpage de `creation_datasets_ML.ipynb`) n'a servi à **aucun** choix : ni variables, ni réglages, ni seuil, ni modèle. **Il est lu une seule fois, ici.**

**Tout est fixé avant la lecture** (`methodologie_ML.md`, sections 10 et 13) :
- **modèle de décision** : le premier du podium de `ml_14`, **CatBoost**, enregistré avec son seuil (`lab_ML/best_models/model_ml_14.joblib`) : entraîné sur tout le train par la boucle, seuil fixé sur les probabilités hors pli du train pour détecter 60 % des défauts (rappel minimal) ;
- **les trois autres modèles** de `ml_14` (à égalité avec CatBoost en validation) sont réentraînés sur le train avec leurs réglages figés et affichés **pour information** : ils ne changent pas la décision ;
- **seuils des autres rappels et bornes des niveaux de risque** : tous calculés sur les probabilités hors pli du **train** (`data/ML/probas_hors_pli/ml_14_v8.csv`), jamais sur le test ;
- **système complet** : la règle du contentieux (définition 2 : tout client au contentieux à M en défaut) pour le contentieux, le modèle pour les autres. Le test ne contient pas de contentieux (retiré avant le découpage) : on lui adjoint un **échantillon de 20 % du contentieux du même périmètre**, tiré comme le test (stratifié sur la cible, `random_state=42`), pour garder les proportions du portefeuille. La règle n'apprend rien : aucune fuite.

**Ce qu'on attend** (validation, `ml_14`) : au rappel de 60 %, une précision des défauts prédits d'environ 27 % et un score décisionnel F2 d'environ 0,485 ; le test doit retrouver ce plafond (section 13 de la méthodologie), avec plus d'incertitude (moins de clients).

**Étapes** :
1. chargement du train, du test et du modèle ; garde-fou de lecture unique ;
2. le modèle de décision sur le test, au seuil fixé, avec son incertitude, face à la validation ;
3. précision à plusieurs rappels (seuils du train), dont le repère de la première itération ;
4. les autres modèles, pour information ;
5. tableau des niveaux de risque et système complet (règle du contentieux + modèle), face à la règle seule.

## 1. Chargement et garde-fou de lecture unique

In [ ]:
import joblib
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score, fbeta_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
dossier_ml = root_dir / "data" / "ML"
CHEMIN_TEST = dossier_ml / "test_dataset.csv"

# Garde-fou : la date de la première lecture du test est notée ; une relance est signalée (les résultats ne doivent plus rien décider)
trace = dossier_ml / "lecture_test.txt"
if trace.exists():
    display(Markdown(f"⚠️ **Le test a déjà été lu** ({trace.read_text(encoding='utf-8').strip()}). Cette relance ne doit servir qu'à réafficher les résultats, jamais à revoir un choix."))
else:
    trace.write_text(f"Première lecture du test : {datetime.now():%d/%m/%Y %H:%M}, version ml_14, modèle de décision CatBoost\n", encoding="utf-8")
    display(Markdown(f"Première lecture du test, notée dans `{trace.name}`."))

train = pd.read_csv(dossier_ml / "train_dataset.csv")
test = pd.read_csv(CHEMIN_TEST)
assert not set(train["ID"]) & set(test["ID"]), "Des clients sont à la fois dans le train et le test"
enregistre = joblib.load(root_dir / "lab_ML" / "best_models" / "model_ml_14.joblib")
assert enregistre["scenario"] == "ml_14" and enregistre["nom"] == "CatBoost"
VARIABLES, SEUIL, RAPPEL_MINIMAL = enregistre["variables"], enregistre["seuil"], enregistre["rappel_minimal"]
y_test = test["dpnm"].to_numpy()
oof = pd.read_csv(dossier_ml / "probas_hors_pli" / "ml_14_v8.csv").set_index("ID").loc[train["ID"]]
y_train = oof["dpnm"].to_numpy()
display(Markdown(f"**Train** : {len(train)} clients, taux de défaut {y_train.mean() * 100:.1f} % ; **test** : {len(test)} clients, "
                 f"taux de défaut {y_test.mean() * 100:.1f} % ; **modèle** : {enregistre['nom']} de `ml_14`, {len(VARIABLES)} variables, "
                 f"seuil {SEUIL:.3f} (rappel minimal {RAPPEL_MINIMAL:.0%}, fixé sur le train)"))

## 2. Le modèle de décision sur le test, au seuil fixé

Le modèle prédit chaque client du test ; un client est signalé si sa probabilité atteint le seuil fixé sur le train. Les mesures sont données avec un **intervalle à 95 %** (2 000 tirages avec remise des clients du test) : le test ne compte qu'environ 770 défauts, son incertitude est plus grande que celle de la validation. En regard : ce qu'annonçait la validation (probabilités hors pli du train, même seuil).

In [ ]:
def mesures(y, p, seuil):
    s = p >= seuil
    return {"Rappel": recall_score(y, s), "Précision des défauts prédits": precision_score(y, s, zero_division=0),
            "Clients signalés": s.mean(), "Score décisionnel F2": fbeta_score(y, s, beta=2),
            "ROC AUC": roc_auc_score(y, p), "Précision moyenne": average_precision_score(y, p)}

p_test = enregistre["modele"].predict_proba(test[VARIABLES])[:, 1]
p_train_oof = oof["CatBoost"].to_numpy()
sur_test = mesures(y_test, p_test, SEUIL)
en_validation = mesures(y_train, p_train_oof, SEUIL)

generateur = np.random.default_rng(42)
tirages = [mesures(y_test[i], p_test[i], SEUIL) for i in (generateur.integers(0, len(y_test), len(y_test)) for _ in range(2000))]
lignes = []
for nom in sur_test:
    valeurs = np.array([t[nom] for t in tirages])
    pourcent = nom not in ("Score décisionnel F2", "ROC AUC", "Précision moyenne")
    f = (lambda v: f"{v * 100:.1f} %") if pourcent else (lambda v: f"{v:.3f}")
    lignes.append({"Mesure": nom, "Validation (train, hors pli)": f(en_validation[nom]), "Test": f(sur_test[nom]),
                   "Intervalle à 95 % (test)": f"{f(np.percentile(valeurs, 2.5))} – {f(np.percentile(valeurs, 97.5))}"})
display(Markdown(f"**CatBoost de `ml_14`, au seuil fixé sur le train ({SEUIL:.3f})**"))
display(pd.DataFrame(lignes).set_index("Mesure"))

s = p_test >= SEUIL
display(Markdown(f"**Sur le test** : {s.sum()} clients signalés, dont {(s & (y_test == 1)).sum()} défauts détectés sur {y_test.sum()} "
                 f"et {(s & (y_test == 0)).sum()} fausses alertes ; {(~s & (y_test == 1)).sum()} défauts manqués."))

**Comment lire** : si les mesures du test tombent dans leur intervalle et près de la validation, le modèle tient sur des clients jamais vus : le plafond mesuré en validation est bien celui des données. Un rappel un peu différent de 60 % est normal : le seuil vient du train, le test a ses propres clients.

## 3. Précision à plusieurs rappels (seuils fixés sur le train)

Pour chaque rappel visé, le seuil est celui qui atteint ce rappel sur les probabilités hors pli du **train** ; il est appliqué tel quel au test. Le rappel de 44 % est le repère de la première itération (précision de 38,5 % sur son test, sur la population laissée au ML) : comparaison indicative, les découpages n'étant pas les mêmes.

In [ ]:
def seuil_pour_rappel(y, p, rappel):
    proba_defauts = np.sort(p[y == 1])[::-1]
    return proba_defauts[int(np.ceil(rappel * len(proba_defauts))) - 1]

lignes = []
for r in (0.30, 0.44, 0.60, 0.70, 0.80):
    seuil = seuil_pour_rappel(y_train, p_train_oof, r)
    v, t = mesures(y_train, p_train_oof, seuil), mesures(y_test, p_test, seuil)
    lignes.append({"Rappel visé (seuil du train)": f"{r:.0%}" + (" (1re itération)" if r == 0.44 else " (rappel minimal)" if r == RAPPEL_MINIMAL else ""),
                   "Seuil": round(seuil, 3), "Rappel sur le test (%)": round(t["Rappel"] * 100, 1),
                   "Précision en validation (%)": round(v["Précision des défauts prédits"] * 100, 1),
                   "Précision sur le test (%)": round(t["Précision des défauts prédits"] * 100, 1),
                   "Clients signalés sur le test (%)": round(t["Clients signalés"] * 100, 1)})
display(pd.DataFrame(lignes).set_index("Rappel visé (seuil du train)"))

## 4. Les autres modèles de `ml_14`, pour information

Régression logistique, RandomForest et MLP, à égalité avec CatBoost en validation : réentraînés sur tout le train avec les réglages retenus par la boucle (relus dans le fichier de résultats), seuil de chacun fixé sur ses probabilités hors pli du train au rappel minimal. **Ces lignes ne changent pas la décision** (CatBoost, fixé avant la lecture) : elles montrent si l'égalité observée en validation se retrouve sur le test.

In [ ]:
import ast
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Variables de ml_14, rangées par encodage (cellule « variables du scénario » de ml_14)
TYPES = {"num": ["LIMIT_BAL"] + [f"PAY_{n}" for n in range(1, 7)] + ["CUMUL_INCIDENT"], "log": [f"PAY_AMT{n}" for n in range(1, 7)],
         "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)], "nom": ["PAY_habituel"]}
assert sorted(sum(TYPES.values(), [])) == sorted(VARIABLES), "Variables différentes du modèle enregistré"
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}
preprocesseur = lambda: ColumnTransformer([(t, TRANSFORMATIONS[t](), cols) for t, cols in TYPES.items()])

def lire_reglage(texte):
    reglage = {}
    for morceau in texte.split(" ; "):
        nom, valeur = morceau.split(" ", 1)
        try:
            reglage[nom] = ast.literal_eval(valeur)
        except (ValueError, SyntaxError):
            reglage[nom] = valeur
    return reglage

resultats = pd.read_csv(dossier_ml / "resultats_scenarios.csv")
lignes_ml14 = resultats[(resultats["scenario"] == "ml_14") & (resultats["version_methode"] == "v8")].set_index("modele")
ESTIMATEURS = {
    "LogisticRegression": lambda r: LogisticRegression(solver="saga", max_iter=5000, class_weight="balanced", random_state=42, **r),
    "RandomForest": lambda r: RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1, **r),
    "MLPClassifier": lambda r: MLPClassifier(max_iter=500, random_state=42, early_stopping=True, **r),
}
lignes = []
for nom, fabrique in ESTIMATEURS.items():
    modele = make_pipeline(preprocesseur(), fabrique(lire_reglage(lignes_ml14.loc[nom, "reglage"]))).fit(train[VARIABLES], train["dpnm"])
    seuil = seuil_pour_rappel(y_train, oof[nom].to_numpy(), RAPPEL_MINIMAL)
    t = mesures(y_test, modele.predict_proba(test[VARIABLES])[:, 1], seuil)
    v = mesures(y_train, oof[nom].to_numpy(), seuil)
    lignes.append({"Modèle": nom, "Réglage": lignes_ml14.loc[nom, "reglage"], "F2 validation": round(v["Score décisionnel F2"], 3),
                   "F2 test": round(t["Score décisionnel F2"], 3), "Rappel test (%)": round(t["Rappel"] * 100, 1),
                   "Précision test (%)": round(t["Précision des défauts prédits"] * 100, 1), "ROC AUC test": round(t["ROC AUC"], 3)})
lignes.insert(0, {"Modèle": "CatBoost (décision)", "Réglage": lignes_ml14.loc["CatBoost", "reglage"], "F2 validation": round(en_validation["Score décisionnel F2"], 3),
                  "F2 test": round(sur_test["Score décisionnel F2"], 3), "Rappel test (%)": round(sur_test["Rappel"] * 100, 1),
                  "Précision test (%)": round(sur_test["Précision des défauts prédits"] * 100, 1), "ROC AUC test": round(sur_test["ROC AUC"], 3)})
display(pd.DataFrame(lignes).set_index("Modèle"))

## 5. Tableau des niveaux de risque et système complet

**Niveaux de risque** (bornes fixées sur le train, probabilités hors pli de CatBoost) :
- **contentieux (règle)** : client au contentieux à M (définition 2), en défaut par la règle ;
- **haut risque** : probabilité au-dessus du seuil qui détecte 30 % des défauts du train (la tête de liste) ;
- **risque modéré** : entre ce seuil et celui du rappel minimal (60 %) ;
- **risque faible** : en dessous.

Pour chaque niveau, sur le test complété de l'échantillon du contentieux : nombre de clients, défauts, taux de défaut (la précision si on signale ce niveau) et part de tous les défauts. Puis le **système complet** (règle + modèle au rappel minimal), face à la **règle seule** (seul le contentieux signalé) : défauts détectés, précision, clients signalés. La banque choisit jusqu'à quel niveau elle signale, selon le coût qu'elle donne à une fausse alerte.

In [ ]:
# Échantillon du contentieux du même périmètre (S12), tiré comme le test : 20 %, stratifié sur la cible, graine 42
niveau_5 = pd.read_csv(root_dir / "data" / "cleaned5_creditcard.csv")
perimetre = niveau_5[(niveau_5["BILL_AMT1"] > 0) & (niveau_5["LIMIT_BAL"] <= 500000)]
contentieux = perimetre[(perimetre["FLAG_CTX"] == 1) & (perimetre["MOIS_SORTIE_CTX"] == -1)]
_, ctx_test = train_test_split(contentieux, test_size=0.2, stratify=contentieux["dpnm"], random_state=42)
assert not set(ctx_test["ID"]) & set(test["ID"])

s30, s60 = seuil_pour_rappel(y_train, p_train_oof, 0.30), SEUIL
niveau = np.where(p_test >= s30, "2. Haut risque", np.where(p_test >= s60, "3. Risque modéré", "4. Risque faible"))
portefeuille = pd.concat([pd.DataFrame({"niveau": "1. Contentieux (règle)", "dpnm": ctx_test["dpnm"].to_numpy()}),
                          pd.DataFrame({"niveau": niveau, "dpnm": y_test})], ignore_index=True)
tableau = portefeuille.groupby("niveau")["dpnm"].agg(Clients="size", Défauts="sum")
tableau["Taux de défaut (%)"] = (tableau["Défauts"] / tableau["Clients"] * 100).round(1)
tableau["Part de tous les défauts (%)"] = (tableau["Défauts"] / tableau["Défauts"].sum() * 100).round(1)
tableau["Défauts cumulés (%)"] = tableau["Part de tous les défauts (%)"].cumsum().round(1)
tableau["Précision cumulée (%)"] = (tableau["Défauts"].cumsum() / tableau["Clients"].cumsum() * 100).round(1)
display(Markdown(f"**Tableau des niveaux de risque** (test : {len(test)} clients ; échantillon du contentieux : {len(ctx_test)} clients ; "
                 f"bornes du train : haut risque dès {s30:.3f}, risque modéré dès {s60:.3f})"))
display(tableau)

y_tout = portefeuille["dpnm"].to_numpy()
strategies = {"Règle seule (contentieux en défaut)": portefeuille["niveau"].eq("1. Contentieux (règle)").to_numpy(),
              "Règle + haut risque": portefeuille["niveau"].isin(["1. Contentieux (règle)", "2. Haut risque"]).to_numpy(),
              "Système complet : règle + modèle au rappel minimal": portefeuille["niveau"].isin(["1. Contentieux (règle)", "2. Haut risque", "3. Risque modéré"]).to_numpy()}
lignes = []
for nom, s in strategies.items():
    lignes.append({"Stratégie": nom, "Défauts détectés": (s & (y_tout == 1)).sum(), "Part des défauts (%)": round((s & (y_tout == 1)).sum() / y_tout.sum() * 100, 1),
                   "Clients signalés": s.sum(), "Part des clients signalés (%)": round(s.mean() * 100, 1),
                   "Précision des défauts prédits (%)": round((s & (y_tout == 1)).sum() / s.sum() * 100, 1), "Fausses alertes": (s & (y_tout == 0)).sum()})
display(Markdown(f"**Système complet face à la règle seule** ({len(portefeuille)} clients, {y_tout.sum()} défauts, taux de défaut {y_tout.mean() * 100:.1f} %)"))
display(pd.DataFrame(lignes).set_index("Stratégie"))

**Comment lire** :
- **Niveaux de risque** : le contentieux concentre une part importante des défauts avec une forte précision ; le haut risque ajoute la tête de liste du modèle ; le risque modéré va chercher plus de défauts, à une précision plus faible. Chaque niveau ajouté coûte plus de bons clients signalés par défaut attrapé : c'est à la banque de dire où elle s'arrête.
- **Système complet** : ce que le modèle ajoute à la règle du contentieux, en défauts détectés et en fausses alertes.
- La conclusion s'écrit après lecture, sans revenir sur aucun choix : le test a parlé une fois.